# SpottingSalmon
Model prediction scipt
- To use final YOLO model to generate fish counts from unseen videos.

_**!Important! Make sure you are connected to a GPU cluster i.e. salmonGPU **_

### Set up:

In [0]:
%pip install ultralytics numpy==1.26.4 --force-reinstall

In [0]:
dbutils.library.restartPython()


In [0]:
import os
import mlflow
from ultralytics import YOLO
import yaml
import pandas as pd
from pyspark.sql.functions import col, current_timestamp

### Step 1: Load best model

In [0]:
# Run ID and artifact relative path
run_id = "95806984982941ebbedc746d54719cb7"
artifact_path = "weights/best.pt"

#model_path = f"runs:/{run_id}/{artifact_path}"
# Download to local temp dir
model_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path=artifact_path
)

print(f"Model downloaded to: {model_path}")

# Load the model
model = YOLO(model_path)

### Step 2: Customise tracker
Generate a tracker so that the same fish in multiple frames is identified as one fish.

In [0]:
# Name of the folder within the filestore directory where the yaml file can be stored
YAML_FOLDER = ""

yaml_content = """
tracker_type: bytetrack 
track_high_thresh: 0.25 # First-stage match threshold
track_low_thresh: 0.1 # Second-stage threshold for low-score matches
new_track_thresh: 0.25 # (float) Start a new track if no match ≥ this
track_buffer: 5 # (int)
match_thresh: 1.0 # (float) Association similarity threshold (IoU/cost); tune with detector quality
fuse_score: False # (bool) Fuse detection score with motion/IoU for matching; stabilizes weak detections
max_lost: 10 
min_box_area: 20
"""

yaml_path = f"/dbfs/FileStore/{YAML_FOLDER}/bytetrack_fish.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"Custom ByteTrack YAML saved at: {yaml_path}")


In [0]:
with open(f"/dbfs/FileStore/{YAML_FOLDER}/bytetrack_fish.yaml") as f:
    config = yaml.safe_load(f)
print(config)


### Step 3: Use model and tracker to predict counts on unseen videos

In [0]:
# Name of the folder within the input directory where the video data sits
INPUT_FOLDER = ""

# Use .predict to infer on a fish image
results = model.predict(f"/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/{INPUT_FOLDER}/Fish (1).jpg")

In [0]:
# Name of the folder within the base directory where the labelled images sit
BASE_FOLDER = ""

# Get directories
video_dir = f"/dbfs/FileStore/{YAML_FOLDER}"
output_summary_file = f"/dbfs/mnt/lab/unrestricted/{BASE_FOLDER}/videos/unseen/fish_tracking_summary.csv"

# Initialize summary
results_summary = []

# Process each video
for video in os.listdir(video_dir):
    if not video.endswith(".mp4"):
        continue

    path = os.path.join(video_dir, video)
    print(f"Processing: {video}")

    # Run ByteTrack tracking
    results = model.track(
        source=path,
        conf=0.15,  # lower to catch small/fast fish
        tracker=f"/dbfs/FileStore/{YAML_FOLDER}/bytetrack_fish.yaml",
        save=True,
        device=0
    )

    # Collect unique fish IDs
    fish_ids = set()
    for r in results:
        if hasattr(r.boxes, "id") and r.boxes.id is not None:
            ids = r.boxes.id.cpu().numpy().tolist()
            fish_ids.update(ids)

    results_summary.append({
        "video": video,
        "fish_count": len(fish_ids)
    })

# Save summary to CSV
df_summary = pd.DataFrame(results_summary)
df_summary.to_csv(output_summary_file, index=False)

print(f"Summary saved to: {output_summary_file}")
print(df_summary)


In [0]:
# Read CSV with pandas
df = pd.read_csv(f"/dbfs/mnt/lab/unrestricted/{BASE_FOLDER}/videos/unseen/fish_tracking_summary.csv")

# Display in Databricks notebook
display(df)


### Save to UC

In [0]:
from pyspark.sql.functions import col, current_timestamp

# Read the CSV you just created
df_summary = pd.read_csv("/dbfs/mnt/lab/unrestricted/{BASE_FOLDER}/videos/unseen/fish_tracking_summary.csv")

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df_summary)

# Ensure 'fish_count' is integer
spark_df = spark_df.withColumn("fish_count", col("fish_count").cast("int"))


In [0]:
TABLE_NAME = "prd_dash_lab.dash_data_science_unrestricted.fish_tracking_summary"

# Create the Delta table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    video STRING,
    fish_count INT,
    ingestion_ts TIMESTAMP
)
USING DELTA
""")


In [0]:
spark_df = spark_df.withColumn("ingestion_ts", current_timestamp())

spark_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TABLE_NAME)
